# ELT Pipeline – Silver Layer (Cleaning & Enrichment)

**Author:** Jarosław Błaziński  
**Project:** E-Commerce Medallion Architecture (Databricks Community Edition)  
**Stack:** PySpark, Delta Tables, Databricks  

---

## Co robi ta warstwa?

Silver = dane oczyszczone i wzbogacone, gotowe do analizy.  
Usuwamy błędy z Bronze, dodajemy kontekst biznesowy i liczymy pochodne metryki.

```
CSV (source)
     ↓
[BRONZE]  – surowe dane
     ↓
[SILVER]  ← jesteśmy tutaj
     ↓
[GOLD]
```

## 1. Konfiguracja

In [ ]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp,
    to_date, month, year, dayofweek,
    when, broadcast, count, sum as spark_sum,
    avg, min as spark_min, max as spark_max
)

spark = SparkSession.builder.appName("ELT_Silver").getOrCreate()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("SILVER")

BRONZE_PATH = "/tmp/medallion/bronze/transactions"
SILVER_PATH = "/tmp/medallion/silver/transactions"

logger.info("Konfiguracja Silver zaladowana")

## 2. Wczytanie danych z Bronze

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)

print(f"Wierszy w Bronze : {df_bronze.count():,}")
df_bronze.show(5, truncate=False)

## 3. Quality check PRZED czyszczeniem

Dokumentujemy stan wejściowy – ważne dla audytu i porównania z wynikiem po czyszczeniu.

In [ ]:
total_before = df_bronze.count()

print("=== Quality Report - PRZED czyszczeniem ===")
df_bronze.agg(
    count("*").alias("total_rows"),
    spark_sum(col("amount").isNull().cast("int")).alias("null_amounts"),
    spark_sum((col("amount") < 0).cast("int")).alias("negative_amounts"),
    spark_sum((col("status") == "cancelled").cast("int")).alias("cancelled"),
    spark_sum((col("status") == "refunded").cast("int")).alias("refunded"),
).show()

## 4. Czyszczenie danych

Usuwamy rekordy które nie nadają się do analizy sprzedaży:  
- NULL w `amount` – brak kwoty = nie możemy liczyć przychodu  
- ujemny `amount` – to zwroty, trafią do osobnej tabeli  
- status `cancelled` – anulowane zamówienia nie są sprzedażą

In [ ]:
df_clean = (
    df_bronze
    .filter(col("amount").isNotNull())
    .filter(col("amount") > 0)
    .filter(col("status") != "cancelled")
    .drop("_ingested_at", "_source_file")
)

total_after = df_clean.count()
retention   = total_after / total_before * 100

print(f"Przed czyszczeniem : {total_before:,}")
print(f"Po czyszczeniu     : {total_after:,}")
print(f"Retencja           : {retention:.1f}%")

## 5. Wzbogacenie o dane wymiarowe (country dimension)

Dołączamy pełne nazwy krajów i stawki VAT per kraj.  
Używamy broadcast join – country_dim jest mała (~4 wiersze), więc broadcast eliminuje shuffle.

In [ ]:
country_dim = spark.createDataFrame(
    [
        ("PL", "Poland",  0.23),
        ("DE", "Germany", 0.19),
        ("DK", "Denmark", 0.25),
        ("SE", "Sweden",  0.25),
    ],
    ["country", "country_name", "vat_rate"]
)

df_enriched = df_clean.join(broadcast(country_dim), on="country", how="left")
df_enriched.show(5, truncate=False)

## 6. Transformacje – metryki pochodne

Obliczamy kolumny potrzebne w analizie:  
- `amount_vat` – kwota z VAT (inna stawka per kraj)  
- `revenue` – przychód = amount * quantity  
- `month`, `year`, `is_weekend` – dla analizy czasowej

In [ ]:
df_silver = (
    df_enriched
    .withColumn("amount_vat",
                col("amount") * (1 + col("vat_rate")))
    .withColumn("revenue",
                col("amount") * col("quantity"))
    .withColumn("revenue_vat",
                col("amount_vat") * col("quantity"))
    .withColumn("transaction_date_only",
                to_date(col("transaction_date")))
    .withColumn("month",
                month(col("transaction_date")))
    .withColumn("year",
                year(col("transaction_date")))
    .withColumn("day_of_week",
                dayofweek(col("transaction_date")))
    .withColumn("is_weekend",
                when(col("day_of_week").isin(1, 7), True).otherwise(False))
    .withColumn("_processed_at", current_timestamp())
)

df_silver.show(5, truncate=False)
df_silver.printSchema()

## 7. Zapis jako Delta Table

In [ ]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("country", "year")
    .save(SILVER_PATH)
)

logger.info("Silver Delta Table zapisana w: %s", SILVER_PATH)
print(f"Silver zapisana: {SILVER_PATH}")

## 8. Time Travel – weryfikacja wersji

In [ ]:
from delta.tables import DeltaTable

delta_silver = DeltaTable.forPath(spark, SILVER_PATH)
delta_silver.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

# Odczyt konkretnej wersji (Time Travel)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_PATH)
print(f"Wierszy w wersji 0: {df_v0.count():,}")

## 9. Quality check PO czyszczeniu

In [ ]:
print("=== Quality Report - PO czyszczeniu ===")
df_silver.agg(
    count("*").alias("total_rows"),
    spark_sum(col("amount").isNull().cast("int")).alias("null_amounts"),
    avg("amount").alias("avg_amount"),
    spark_min("amount").alias("min_amount"),
    spark_max("amount").alias("max_amount"),
    avg("revenue").alias("avg_revenue"),
).show()

print("Rozklad po miesiacach:")
df_silver.groupBy("month").count().orderBy("month").show()

print("Rozklad po metodach platnosci:")
df_silver.groupBy("payment_method").count().orderBy("count", ascending=False).show()

## Podsumowanie Silver Layer

| Krok | Status |
|---|---|
| Wczytanie z Bronze Delta Table | ✅ |
| Quality check przed czyszczeniem | ✅ |
| Usunięcie NULLi, zwrotów, anulowanych | ✅ |
| Broadcast join z country dimension | ✅ |
| Obliczenie VAT, revenue, metryk czasowych | ✅ |
| Zapis Delta Table (partitioned by country, year) | ✅ |
| Time Travel weryfikacja | ✅ |

**Następny krok:** `6_ELT_Gold.ipynb` – agregaty, ranking i eksport do Power BI.